# Pengolahan Data Genangan Damkar Surabaya

Notebook ini mengolah data historis genangan dari Dinas Pemadam Kebakaran dan
Penyelamatan Kota Surabaya (file `DATA_GENANGAN_2025-2026.xlsx`, sheet
`exportOverall`, 409 record). Dua tahap utama:

1. **Parsing kedalaman air** dari kolom `Detail_Objek Evakuasi` menggunakan
   regex, karena format teks bervariasi ("Genangan : 20 cm", "± 15 s/d 20 cm",
   "80cm-20cm", dll.)
2. **Geocoding lokasi** dari nama jalan + kecamatan + kelurahan ke koordinat
   lat/lon menggunakan Nominatim (OpenStreetMap)

**Input:** `DATA_GENANGAN_2025-2026.xlsx`

**Output:**
- `damkar_genangan_geocoded.csv` (seluruh 409 record dengan koordinat dan kedalaman parsed)
- `damkar_geocoding_report.csv` (detail status geocoding per titik, untuk transparansi)
- `damkar_cluster_summary.csv` (ringkasan cluster event banjir per tanggal)


In [8]:
%pip install -q openpyxl pandas requests tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import re
import time
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import requests
from tqdm import tqdm

# ===== KONFIGURASI =====
INPUT_FILE = "../backend/app/data/data-genangan-25-26.xlsx"
SHEET_NAME = "exportOverall"
OUTPUT_DIR = Path("output_damkar")
OUTPUT_DIR.mkdir(exist_ok=True)

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
USER_AGENT = "MDCVRP-IF-SI-TA/1.0 (student research; Surabaya flood pumping)"
GEOCODE_DELAY_S = 1.5  # Nominatim policy: max 1 request per second

SBY_LAT_MIN, SBY_LAT_MAX = -7.40, -7.20
SBY_LON_MIN, SBY_LON_MAX = 112.58, 112.87

print("Setup selesai.")

Setup selesai.


## 1. Muat Data dari Excel

In [ ]:
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
print(f"Total record: {len(df)}")
print(f"Kolom: {list(df.columns)}")
print()

# Rename kolom untuk konsistensi
df = df.rename(columns={
    "NO": "no",
    "Info_Waktu": "tanggal",
    "Jalan": "jalan",
    "Kecamatan": "kecamatan",
    "Kelurahan": "kelurahan",
    "Area": "area",
    "Objek Evakuasi": "objek",
    "Detail_Objek Evakuasi": "detail",
    "Terima_Berita": "terima_berita",
    "Berangkat": "berangkat",
    "TibaTKP": "tiba_tkp",
    "Penanganan Evakuasi": "mulai_penanganan",
    "Selesai": "selesai",
    "TibaPangkalan": "tiba_pangkalan",
    "RESPON TIME": "respon_time",
})

# Pastikan tanggal dalam format datetime
df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")
df = df.dropna(subset=["tanggal"]).reset_index(drop=True)

# datetime = tanggal (date) + jam dari terima_berita, mis. "2026-06-22 05:14:00"
df["terima_berita"] = pd.to_datetime(df["terima_berita"], errors="coerce")
time_of_day = df["terima_berita"] - df["terima_berita"].dt.normalize()   # jam:menit:detik saja
df["datetime"] = df["tanggal"].dt.normalize() + time_of_day.fillna(pd.Timedelta(0))

print(f"Record valid: {len(df)}")
print(f"Rentang tanggal: {df['tanggal'].min().date()} s/d {df['tanggal'].max().date()}")
df.head()

## 2. Parsing Kedalaman Air dari Teks

Kolom `detail` berisi teks bebas dengan format yang bervariasi. Contoh:
- `"Genangan : 20 cm"`
- `"Genangan : 20 s/d 30 cm"` atau `"± 15 s/d 20 cm"`
- `"80cm-20cm"` (format debit: max-min)
- `"Jl. Raya Kendangsari ( RSIA )= 15cm"`
- `"Jl. S.Memet (SMA 3) : 50cm"`
- `"Kenpark : 25 cm - 10cm"`

Strategi: ekstrak SEMUA angka yang diikuti "cm" dari teks, lalu ambil
nilai **maksimum** sebagai representasi kedalaman terburuk di titik tersebut.
Ini konsisten dengan pendekatan "nilai tertinggi" yang dipakai untuk
klasifikasi jalan (C2).

In [11]:
def parse_depth_cm(text):
    """Ekstrak kedalaman air (cm) dari teks deskriptif.

    Strategi:
    1. Cari semua angka yang diikuti 'cm' (dengan atau tanpa spasi)
    2. Cari angka dalam pola 'X s/d Y', 'X - Y', 'X sampai Y'
    3. Ambil nilai maksimum dari semua angka yang ditemukan
    4. Return None jika tidak ada angka ditemukan
    """
    if not isinstance(text, str) or not text.strip():
        return None

    text = text.lower().strip()
    numbers = []

    # Pattern 1: angka diikuti 'cm' (dengan atau tanpa spasi, titik, koma)
    # Contoh: "50cm", "50 cm", "15-20 cm"
    for m in re.finditer(r'(\d+(?:[.,]\d+)?)\s*cm', text):
        try:
            val = float(m.group(1).replace(',', '.'))
            if 0 < val <= 500:  # sanity check: 0-500 cm
                numbers.append(val)
        except ValueError:
            pass

    # Pattern 2: angka sebelum 's/d', 'sampai', '-' yang diikuti angka + cm
    # Contoh: "15 s/d 20 cm" -> ambil 15 juga (pattern 1 sudah tangkap 20)
    for m in re.finditer(r'(\d+(?:[.,]\d+)?)\s*(?:s/?d|sampai|s\.d\.?|hingga)\s*(\d+)', text):
        try:
            val1 = float(m.group(1).replace(',', '.'))
            val2 = float(m.group(2).replace(',', '.'))
            if 0 < val1 <= 500:
                numbers.append(val1)
            if 0 < val2 <= 500:
                numbers.append(val2)
        except ValueError:
            pass

    # Pattern 3: "80cm-20cm" (debit format)
    for m in re.finditer(r'(\d+)\s*cm\s*[-]\s*(\d+)\s*cm', text):
        try:
            val1 = float(m.group(1))
            val2 = float(m.group(2))
            if 0 < val1 <= 500:
                numbers.append(val1)
            if 0 < val2 <= 500:
                numbers.append(val2)
        except ValueError:
            pass

    if numbers:
        return max(numbers)  # ambil nilai maksimum

    # Pattern 4: angka standalone (tanpa 'cm') sebagai fallback terakhir
    # Hanya kalau ada kata 'genangan' di teks
    if 'genangan' in text or 'banjir' in text:
        standalone = re.findall(r'(\d+(?:[.,]\d+)?)', text)
        for s in standalone:
            try:
                val = float(s.replace(',', '.'))
                if 5 <= val <= 200:  # range masuk akal untuk cm
                    numbers.append(val)
            except ValueError:
                pass
        if numbers:
            return max(numbers)

    return None


# Test pada sampel data
test_cases = [
    "Genangan : 20 cm",
    "Genangan : 20 s/d 30 cm",
    "± 15 s/d 20 cm",
    "Debit air 80cm-20cm",
    "Jl. Raya Kendangsari ( RSIA )= 15cm",
    "Jl. S.Memet (SMA 3) : 50cm",
    "Kenpark : 25 cm - 10cm",
    "Jl. Simorejo sari B = ± 15 s/d 20 cm",
    "Genangan : ± 15 cm",
    "Genangan : 70 cm",
    "Genangan : 10 cm",
]

print("=== Test Parsing ===")
for tc in test_cases:
    print(f"  '{tc}' -> {parse_depth_cm(tc)} cm")

=== Test Parsing ===
  'Genangan : 20 cm' -> 20.0 cm
  'Genangan : 20 s/d 30 cm' -> 30.0 cm
  '± 15 s/d 20 cm' -> 20.0 cm
  'Debit air 80cm-20cm' -> 80.0 cm
  'Jl. Raya Kendangsari ( RSIA )= 15cm' -> 15.0 cm
  'Jl. S.Memet (SMA 3) : 50cm' -> 50.0 cm
  'Kenpark : 25 cm - 10cm' -> 25.0 cm
  'Jl. Simorejo sari B = ± 15 s/d 20 cm' -> 20.0 cm
  'Genangan : ± 15 cm' -> 15.0 cm
  'Genangan : 70 cm' -> 70.0 cm
  'Genangan : 10 cm' -> 10.0 cm


In [12]:
# Terapkan parsing ke seluruh dataset
df["depth_cm"] = df["detail"].apply(parse_depth_cm)

parsed_ok = df["depth_cm"].notna().sum()
parsed_fail = df["depth_cm"].isna().sum()
print(f"Parsing berhasil: {parsed_ok} ({100*parsed_ok/len(df):.1f}%)")
print(f"Parsing gagal   : {parsed_fail} ({100*parsed_fail/len(df):.1f}%)")
print()

if parsed_fail > 0:
    print("Record yang gagal di-parse:")
    for _, row in df[df["depth_cm"].isna()].iterrows():
        print(f"  No.{row['no']}: '{row['detail']}'")

print()
print("Distribusi kedalaman (cm):")
print(df["depth_cm"].describe())

Parsing berhasil: 409 (100.0%)
Parsing gagal   : 0 (0.0%)


Distribusi kedalaman (cm):
count    409.000000
mean      20.136919
std       14.667251
min        2.000000
25%       10.000000
50%       15.000000
75%       20.000000
max      100.000000
Name: depth_cm, dtype: float64


## 3. Geocoding Lokasi

Setiap record memiliki nama jalan, kecamatan, dan kelurahan, tapi TIDAK
memiliki koordinat lat/lon. Geocoding dilakukan menggunakan Nominatim
(OpenStreetMap) dengan query bertingkat:

1. **Query lengkap**: "Jl. [nama jalan], [kelurahan], [kecamatan], Surabaya"
2. **Query tanpa kelurahan** (fallback 1): "[nama jalan], [kecamatan], Surabaya"
3. **Query kecamatan saja** (fallback 2): "[kecamatan], Surabaya"

Setiap query dibatasi ke bounding box Surabaya untuk menghindari hasil
di luar kota. Koordinat hasil geocoding divalidasi terhadap bounding box.

In [13]:
def geocode_nominatim(query, bbox=None, max_retries=2):
    """Geocode satu query via Nominatim. Return (lat, lon) atau (None, None)."""
    headers = {"User-Agent": USER_AGENT}
    params = {
        "q": query,
        "format": "json",
        "limit": 1,
        "countrycodes": "id",
    }
    if bbox:
        # viewbox = lon_min, lat_min, lon_max, lat_max
        params["viewbox"] = f"{bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]}"
        params["bounded"] = 1

    for attempt in range(max_retries):
        try:
            r = requests.get(NOMINATIM_URL, params=params, headers=headers, timeout=15)
            if r.status_code == 200:
                data = r.json()
                if data:
                    lat = float(data[0]["lat"])
                    lon = float(data[0]["lon"])
                    return lat, lon
            elif r.status_code == 429:
                time.sleep(5)
        except Exception:
            time.sleep(3)
    return None, None


def geocode_tiered(jalan, kecamatan, kelurahan):
    """Geocoding bertingkat: coba query paling spesifik dulu, lalu fallback."""
    bbox = (SBY_LAT_MIN, SBY_LON_MIN, SBY_LAT_MAX, SBY_LON_MAX)

    # Bersihkan nama jalan
    jalan_clean = str(jalan).strip() if pd.notna(jalan) else ""
    kec_clean = str(kecamatan).strip() if pd.notna(kecamatan) else ""
    kel_clean = str(kelurahan).strip() if pd.notna(kelurahan) else ""

    # Level 1: query lengkap
    if jalan_clean and kel_clean and kec_clean:
        q = f"{jalan_clean}, {kel_clean}, {kec_clean}, Surabaya, Jawa Timur"
        lat, lon = geocode_nominatim(q, bbox)
        if lat is not None:
            return lat, lon, "level1_full"
        time.sleep(GEOCODE_DELAY_S)

    # Level 2: tanpa kelurahan
    if jalan_clean and kec_clean:
        q = f"{jalan_clean}, {kec_clean}, Surabaya, Jawa Timur"
        lat, lon = geocode_nominatim(q, bbox)
        if lat is not None:
            return lat, lon, "level2_no_kel"
        time.sleep(GEOCODE_DELAY_S)

    # Level 3: jalan + Surabaya saja
    if jalan_clean:
        q = f"{jalan_clean}, Surabaya, Jawa Timur"
        lat, lon = geocode_nominatim(q, bbox)
        if lat is not None:
            return lat, lon, "level3_jalan_only"
        time.sleep(GEOCODE_DELAY_S)

    # Level 4: kelurahan + kecamatan (centroid kelurahan)
    if kel_clean and kec_clean:
        q = f"{kel_clean}, {kec_clean}, Surabaya, Jawa Timur"
        lat, lon = geocode_nominatim(q, bbox)
        if lat is not None:
            return lat, lon, "level4_kelurahan"
        time.sleep(GEOCODE_DELAY_S)

    # Level 5: kecamatan saja (centroid kecamatan)
    if kec_clean:
        q = f"{kec_clean}, Surabaya, Jawa Timur"
        lat, lon = geocode_nominatim(q, bbox)
        if lat is not None:
            return lat, lon, "level5_kecamatan"
        time.sleep(GEOCODE_DELAY_S)

    return None, None, "failed"


print("Fungsi geocoding siap.")
print(f"Estimasi waktu: {len(df) * GEOCODE_DELAY_S / 60:.0f} menit (worst case)")

Fungsi geocoding siap.
Estimasi waktu: 10 menit (worst case)


In [14]:
# Jalankan geocoding untuk seluruh record
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Geocoding"):
    lat, lon, level = geocode_tiered(row["jalan"], row["kecamatan"], row["kelurahan"])

    # Validasi: koordinat harus dalam bounding box Surabaya
    if lat is not None:
        if not (SBY_LAT_MIN <= lat <= SBY_LAT_MAX and SBY_LON_MIN <= lon <= SBY_LON_MAX):
            lat, lon, level = None, None, "out_of_bbox"

    results.append({
        "no": row["no"],
        "lat": lat,
        "lon": lon,
        "geocode_level": level,
        "query_jalan": row["jalan"],
        "query_kecamatan": row["kecamatan"],
        "query_kelurahan": row["kelurahan"],
    })

    time.sleep(GEOCODE_DELAY_S)

geocode_df = pd.DataFrame(results)

# Statistik geocoding
level_counts = geocode_df["geocode_level"].value_counts()
print("\n=== Statistik Geocoding ===")
for level, count in level_counts.items():
    print(f"  {level}: {count} ({100*count/len(geocode_df):.1f}%)")

n_success = geocode_df["lat"].notna().sum()
n_fail = geocode_df["lat"].isna().sum()
print(f"\nTotal berhasil: {n_success} ({100*n_success/len(geocode_df):.1f}%)")
print(f"Total gagal   : {n_fail} ({100*n_fail/len(geocode_df):.1f}%)")

Geocoding: 100%|██████████| 409/409 [47:50<00:00,  7.02s/it]  



=== Statistik Geocoding ===
  level1_full: 229 (56.0%)
  level4_kelurahan: 171 (41.8%)
  level3_jalan_only: 6 (1.5%)
  level2_no_kel: 3 (0.7%)

Total berhasil: 409 (100.0%)
Total gagal   : 0 (0.0%)


## 4. Gabung dan Simpan

In [15]:
# Merge geocoding results ke dataframe utama
df["lat"] = geocode_df["lat"]
df["lon"] = geocode_df["lon"]
df["geocode_level"] = geocode_df["geocode_level"]

# Simpan seluruh data (termasuk yang gagal geocoding, untuk review manual)
df.to_csv(OUTPUT_DIR / "damkar_genangan_geocoded.csv", index=False)
geocode_df.to_csv(OUTPUT_DIR / "damkar_geocoding_report.csv", index=False)

print(f"Tersimpan: {OUTPUT_DIR}/damkar_genangan_geocoded.csv ({len(df)} rows)")
print(f"Tersimpan: {OUTPUT_DIR}/damkar_geocoding_report.csv")
print()

# Preview data yang berhasil
success = df[df["lat"].notna()]
print(f"Record dengan koordinat: {len(success)}")
print(success[["no", "tanggal", "jalan", "kecamatan", "depth_cm", "lat", "lon"]].head(10))

Tersimpan: output_damkar/damkar_genangan_geocoded.csv (409 rows)
Tersimpan: output_damkar/damkar_geocoding_report.csv

Record dengan koordinat: 409
   no    tanggal                                            jalan  \
0   1 2025-01-01                                      Jl. Srikana   
1   2 2025-01-01              Jl. Ngagel Jaya Utara (Gereja SMTB)   
2   3 2025-01-01                  Jl. Ngagel Kebonsari ( SMP 12 )   
3   4 2025-01-01                             Jl. Tenggilis Mejoyo   
4   5 2025-01-01                              Jl. S.Memet (SMA 3)   
5   6 2025-01-01                                          Kenpark   
6   7 2025-01-05                              Jl. Simorejo sari B   
7   8 2025-01-05         Rusun Warugunung ( depan lapangan bola )   
8   9 2025-01-11                    Jl. Raya Kendangsari ( RSIA )   
9  10 2025-01-11  Gereja SMTB ( sisi barat ) Jl.Ngagel Jaya Utara   

          kecamatan  depth_cm       lat         lon  
0            Gubeng      80.0 -7.27412

## 5. Analisis Cluster Event Banjir

Identifikasi cluster event banjir berdasarkan tanggal berurutan (gap
maksimal 2 hari dianggap satu event). Cluster ini menjadi kandidat skenario
statis untuk model MDCVRP-IF-SI.

In [16]:
# Hanya gunakan record yang berhasil di-geocoding
geo_ok = df[df["lat"].notna()].copy()
geo_ok["date"] = geo_ok["tanggal"].dt.date

# Hitung jumlah titik per tanggal
date_counts = geo_ok.groupby("date").size().reset_index(name="n_points")
date_counts = date_counts.sort_values("date")

# Identifikasi cluster (gap <= 2 hari)
clusters = []
current_cluster = [date_counts.iloc[0]]

for i in range(1, len(date_counts)):
    prev_date = date_counts.iloc[i - 1]["date"]
    curr_date = date_counts.iloc[i]["date"]
    if (curr_date - prev_date).days <= 2:
        current_cluster.append(date_counts.iloc[i])
    else:
        clusters.append(current_cluster)
        current_cluster = [date_counts.iloc[i]]
clusters.append(current_cluster)

# Ringkasan cluster
cluster_summary = []
for ci, cl in enumerate(clusters):
    dates_in_cluster = [r["date"] for r in cl]
    total = sum(r["n_points"] for r in cl)
    cluster_summary.append({
        "cluster_id": ci + 1,
        "start_date": min(dates_in_cluster),
        "end_date": max(dates_in_cluster),
        "n_days": len(dates_in_cluster),
        "n_points": total,
        "dates_detail": "; ".join(f"{r['date']}({r['n_points']})" for r in cl),
    })

cluster_df = pd.DataFrame(cluster_summary)
cluster_df = cluster_df.sort_values("n_points", ascending=False).reset_index(drop=True)

print("=== CLUSTER EVENT BANJIR (diurutkan berdasarkan jumlah titik) ===")
print()
for _, row in cluster_df.head(10).iterrows():
    print(f"Cluster {row['cluster_id']}: {row['start_date']} s/d {row['end_date']}")
    print(f"  {row['n_days']} hari, {row['n_points']} titik genangan")
    print(f"  Detail: {row['dates_detail']}")
    print()

cluster_df.to_csv(OUTPUT_DIR / "damkar_cluster_summary.csv", index=False)
print(f"Tersimpan: {OUTPUT_DIR}/damkar_cluster_summary.csv")

=== CLUSTER EVENT BANJIR (diurutkan berdasarkan jumlah titik) ===

Cluster 44: 2026-06-22 s/d 2026-06-23
  2 hari, 34 titik genangan
  Detail: 2026-06-22(19); 2026-06-23(15)

Cluster 3: 2025-01-11 s/d 2025-01-15
  4 hari, 31 titik genangan
  Detail: 2025-01-11(13); 2025-01-12(10); 2025-01-13(7); 2025-01-15(1)

Cluster 21: 2025-11-09 s/d 2025-11-12
  4 hari, 29 titik genangan
  Detail: 2025-11-09(5); 2025-11-10(9); 2025-11-11(3); 2025-11-12(12)

Cluster 26: 2025-12-23 s/d 2025-12-27
  4 hari, 27 titik genangan
  Detail: 2025-12-23(19); 2025-12-24(1); 2025-12-25(2); 2025-12-27(5)

Cluster 12: 2025-03-13 s/d 2025-03-17
  5 hari, 25 titik genangan
  Detail: 2025-03-13(3); 2025-03-14(11); 2025-03-15(9); 2025-03-16(1); 2025-03-17(1)

Cluster 28: 2026-01-02 s/d 2026-01-05
  3 hari, 22 titik genangan
  Detail: 2026-01-02(1); 2026-01-04(17); 2026-01-05(4)

Cluster 7: 2025-02-13 s/d 2025-02-13
  1 hari, 20 titik genangan
  Detail: 2025-02-13(20)

Cluster 38: 2026-03-12 s/d 2026-03-17
  5 hari, 1

In [ ]:
# Export cluster terbesar sebagai skenario siap pakai
# Ambil top 3 cluster sebagai kandidat skenario
for rank, (_, cl_row) in enumerate(cluster_df.head(3).iterrows()):
    start = cl_row["start_date"]
    end = cl_row["end_date"]
    mask = (geo_ok["date"] >= start) & (geo_ok["date"] <= end)
    scenario = geo_ok[mask].copy()

    filename = f"skenario_{rank+1}_{start}_to_{end}_{cl_row['n_points']}pts.csv"
    # Pakai kolom datetime (tanggal + jam) sebagai ganti tanggal date-only
    cols_out = ["no", "datetime", "jalan", "kecamatan", "kelurahan", "area",
                "detail", "depth_cm", "lat", "lon",
                "terima_berita", "tiba_tkp", "mulai_penanganan", "selesai"]
    scenario[cols_out].to_csv(OUTPUT_DIR / filename, index=False)
    print(f"Skenario {rank+1}: {filename} ({len(scenario)} titik)")

    # Distribusi kecamatan dalam skenario ini
    kec = scenario["kecamatan"].value_counts()
    print(f"  Kecamatan: {', '.join(f'{k}({v})' for k, v in kec.head(8).items())}")
    print(f"  Kedalaman: min={scenario['depth_cm'].min()}, "
          f"median={scenario['depth_cm'].median()}, "
          f"max={scenario['depth_cm'].max()} cm")
    print()

## 6. Deduplikasi Lokasi dalam Cluster

Dalam satu cluster event, bisa jadi ada dua record dengan lokasi yang sama
(jalan yang sama, kecamatan sama) tapi tanggal berbeda (banjir berulang
di lokasi yang sama selama event berlangsung). Untuk skenario statis,
lokasi duplikat digabungkan dengan mengambil kedalaman **maksimum**
sebagai representasi kasus terburuk.

In [18]:
def deduplicate_scenario(scenario_df, distance_threshold_m=200):
    """Deduplikasi titik genangan yang terlalu berdekatan.

    Dua titik dianggap duplikat jika jaraknya < threshold meter.
    Dari duplikat, ambil kedalaman maksimum dan simpan satu koordinat.
    """
    from math import radians, sin, cos, sqrt, atan2

    def haversine_m(lat1, lon1, lat2, lon2):
        R = 6_371_000
        dlat = radians(lat2 - lat1)
        dlon = radians(lon2 - lon1)
        a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
        return R * 2 * atan2(sqrt(a), sqrt(1-a))

    df = scenario_df.copy().reset_index(drop=True)
    merged = [False] * len(df)
    result = []

    for i in range(len(df)):
        if merged[i]:
            continue
        group = [i]
        for j in range(i + 1, len(df)):
            if merged[j]:
                continue
            d = haversine_m(df.loc[i, "lat"], df.loc[i, "lon"],
                           df.loc[j, "lat"], df.loc[j, "lon"])
            if d < distance_threshold_m:
                group.append(j)
                merged[j] = True

        subset = df.loc[group]
        representative = subset.iloc[0].copy()
        representative["depth_cm"] = subset["depth_cm"].max()
        representative["n_merged"] = len(group)
        if len(group) > 1:
            representative["merged_from"] = "; ".join(subset["jalan"].astype(str).tolist())
        else:
            representative["merged_from"] = ""
        result.append(representative)

    return pd.DataFrame(result).reset_index(drop=True)


# Terapkan deduplikasi pada top 3 skenario
for rank, (_, cl_row) in enumerate(cluster_df.head(3).iterrows()):
    start = cl_row["start_date"]
    end = cl_row["end_date"]
    mask = (geo_ok["date"] >= start) & (geo_ok["date"] <= end)
    scenario = geo_ok[mask].copy()

    deduped = deduplicate_scenario(scenario, distance_threshold_m=200)

    filename = f"skenario_{rank+1}_dedup_{len(deduped)}pts.csv"
    deduped.to_csv(OUTPUT_DIR / filename, index=False)

    n_merged = (deduped["n_merged"] > 1).sum()
    print(f"Skenario {rank+1}: {len(scenario)} record -> {len(deduped)} titik unik "
          f"({n_merged} lokasi digabungkan)")

Skenario 1: 34 record -> 25 titik unik (9 lokasi digabungkan)
Skenario 2: 31 record -> 23 titik unik (7 lokasi digabungkan)
Skenario 3: 29 record -> 27 titik unik (2 lokasi digabungkan)


## Catatan Penting

- **Geocoding level 4 dan 5** (fallback ke centroid kelurahan/kecamatan)
  menghasilkan koordinat yang kurang presisi. Titik dengan level ini
  sebaiknya diverifikasi manual di Google Maps sebelum dipakai di model.
- **Parsing kedalaman** mengambil nilai maksimum dari semua angka "cm"
  dalam teks. Untuk kasus seperti "80cm-20cm" (debit air dari max ke min),
  nilai 80 cm diambil sebagai representasi kasus terburuk.
- **Deduplikasi** dengan radius 200 meter menggabungkan titik yang secara
  operasional merupakan satu lokasi genangan. Radius ini bisa disesuaikan
  sesuai kebutuhan.
- Setelah skenario dipilih (diskusi dengan dosen), file CSV skenario
  terpilih bisa langsung di-rename jadi `floods.csv` dan ditempatkan
  di `backend/app/data/`. Kolom yang dibutuhkan backend:
  `Latitude`, `Longitude`, `Ketinggian (cm)`, `Deskripsi`, `road_class`,
  `dist_faskes_m`.
- Kolom `road_class` dan `dist_faskes_m` perlu dihitung ulang menggunakan
  Notebook 5 (klasifikasi jalan) dan Notebook 6 (jarak faskes OSRM)
  setelah koordinat titik genangan berubah.
- Matriks jarak (Notebook 4) juga perlu dihitung ulang karena jumlah
  dan posisi node berubah.
